# 07 — Independent steganalysis

Final detectability is evaluated with **two detector classes that are not used to construct the allocation score**: (1) an SRM-inspired residual/co-occurrence logistic detector (`SRM-lite`, explicitly not the full Spatial Rich Model), and (2) a compact residual CNN with fixed high-pass front-end. Both are trained only on train sources, validated on validation sources, and scored on test sources. The detector training mixture uses a deterministic variety of strategies/payloads; final AUC and TPR@5%FPR are reported separately for every frozen strategy/payload.

This notebook can take substantial CPU time; CNN training uses a configurable subset of train images and all final test cases are scored.

In [ ]:
from pathlib import Path
import json, joblib, pandas as pd, numpy as np, yaml
from rdhlab.io import read_gray
from rdhlab.pipeline import load_payload_freeze, prepare_image_context, run_frozen_image_precomputed, deterministic_seed
from rdhlab.detectors import fit_rich_detector, detector_metrics, paired_detector_bootstrap, train_residual_cnn, score_residual_cnn, save_cnn

config=yaml.safe_load(Path('/workspace/config/experiment.yaml').read_text())
seed=int(config['project']['seed']); bs=int(config['dataset']['block_size'])
manifest=pd.read_csv(config['dataset']['prepared_manifest']); train=manifest[manifest.split=='train'].reset_index(drop=True); val=manifest[manifest.split=='validation'].reset_index(drop=True); test=manifest[manifest.split=='test'].reset_index(drop=True)
risk=joblib.load('/workspace/results/models/block_risk.joblib'); payloads=list(map(float,load_payload_freeze('/workspace/config/frozen_payloads.json')['levels']))
alpha=float(json.loads(Path('/workspace/config/frozen_allocator.json').read_text())['alpha']); strategies=list(config['allocator']['strategies'])
model_dir=Path('/workspace/results/detectors'); model_dir.mkdir(parents=True,exist_ok=True)
context_dir=Path('/workspace/results/frozen_test/contexts')
ctx_cache={p.stem:joblib.load(p) for p in context_dir.glob('*.joblib')} if context_dir.exists() else {}
print('Cached test contexts:',len(ctx_cache))

In [ ]:
def mixed_pairs(frame,n_images):
    covers=[]; stegos=[]
    for i,row in frame.head(n_images).iterrows():
        x=read_gray(row.path); sid=str(row.source_id)
        rng=np.random.default_rng(deterministic_seed(seed,'detector_mix',sid))
        strategy=strategies[int(rng.integers(0,len(strategies)))]
        bpp=payloads[int(rng.integers(0,len(payloads)))]
        orders,br,plans=prepare_image_context(x,sid,risk,alpha,bs,seed)
        rr=run_frozen_image_precomputed(x,sid,bpp,strategy,orders,br,bs,seed,plans=plans)
        if rr['feasible']:
            covers.append(x); stegos.append(rr['stego'])
        if (i+1)%100==0: print(i+1,'/',n_images)
    return covers,stegos

rich_n=min(int(config['detectors']['rich_train_images']),len(train))
rich_c,rich_s=mixed_pairs(train,rich_n)
rich=fit_rich_detector(rich_c,rich_s,seed)
joblib.dump(rich,model_dir/'srm_lite.joblib')
print('Rich detector training pairs:',len(rich_c))

In [ ]:
cnn_n=min(int(config['detectors']['cnn_train_images']),len(rich_c)); cnn_v=min(int(config['detectors']['cnn_validation_images']),len(val))
# Reuse the first train pairs already generated for the rich detector.
cnn_c,cnn_s=rich_c[:cnn_n],rich_s[:cnn_n]
val_c,val_s=mixed_pairs(val,cnn_v)
cnn,history,device=train_residual_cnn(cnn_c,cnn_s,(val_c,val_s),epochs=int(config['detectors']['cnn_epochs']),batch_size=int(config['detectors']['cnn_batch_size']),seed=seed)
save_cnn(cnn,model_dir/'residual_cnn.pt',{'history':history,'device':device,'train_pairs':len(cnn_c),'validation_pairs':len(val_c)})
print('CNN device:',device); display(pd.DataFrame(history))

In [ ]:
# Final test scoring, separately for each method and payload.
# Cover detector scores are invariant across methods/payloads, so compute them once.
cover_images=[read_gray(p) for p in test.path]
cover_rich=np.asarray([rich.score(x) for x in cover_images],dtype=float)
cover_cnn=score_residual_cnn(cnn,cover_images,device=device,batch_size=int(config['detectors']['cnn_batch_size']))
print('Precomputed detector scores for',len(cover_images),'test covers')

rows=[]
for strategy in strategies:
    for bpp in payloads:
        cover_idx=[]; rich_stego=[]; cnn_stego_images=[]
        for i,row in test.iterrows():
            x=cover_images[i]; sid=str(row.source_id)
            if sid in ctx_cache: orders,br,plans=ctx_cache[sid]
            else: orders,br,plans=prepare_image_context(x,sid,risk,alpha,bs,seed)
            rr=run_frozen_image_precomputed(x,sid,bpp,strategy,orders,br,bs,seed,plans=plans)
            if not rr['feasible']: continue
            cover_idx.append(i); rich_stego.append(rich.score(rr['stego'])); cnn_stego_images.append(rr['stego'])
            if (i+1)%250==0: print(strategy,bpp,i+1,'/2000')
        cover_idx=np.asarray(cover_idx,dtype=int)
        rc=cover_rich[cover_idx]; rs=np.asarray(rich_stego,dtype=float)
        cc=cover_cnn[cover_idx]; cs=score_residual_cnn(cnn,cnn_stego_images,device=device,batch_size=int(config['detectors']['cnn_batch_size']))
        for name,c_scores,s_scores in [('srm_lite',rc,rs),('residual_cnn',cc,cs)]:
            scores=np.column_stack([c_scores,s_scores]).reshape(-1); y=np.tile([0,1],len(c_scores))
            m=detector_metrics(y,scores,float(config['detectors']['fixed_fpr']))
            boot=paired_detector_bootstrap(c_scores,s_scores,float(config['detectors']['fixed_fpr']),n_resamples=int(config['statistics']['cluster_bootstrap_resamples']),confidence=float(config['statistics']['confidence']),seed=seed)
            m.update(boot)
            m.update({'detector':name,'strategy':strategy,'target_net_bpp':bpp,'pairs':len(c_scores)}); rows.append(m)
metrics=pd.DataFrame(rows); metrics.to_csv(model_dir/'test_detectability.csv',index=False); display(metrics)

In [ ]:
# Adaptive/retrained detector check at the primary (middle) payload.
# This is separate from the universal detectors above and tests whether the
# allocation advantage survives a detector retrained specifically for each method.
if bool(config['detectors']['adaptive_primary_payload_only']):
    primary_bpp=payloads[len(payloads)//2]
    adaptive_rows=[]
    n_adapt=min(int(config['detectors']['adaptive_train_images']),len(train))
    for strategy in strategies:
        ac=[]; ast=[]
        for i,row in train.head(n_adapt).iterrows():
            x=read_gray(row.path); sid=str(row.source_id)
            orders,br,plans=prepare_image_context(x,sid,risk,alpha,bs,seed)
            rr=run_frozen_image_precomputed(x,sid,primary_bpp,strategy,orders,br,bs,seed,plans=plans)
            if rr['feasible']: ac.append(x); ast.append(rr['stego'])
        det=fit_rich_detector(ac,ast,seed)
        y=[]; scores=[]
        for i,row in test.iterrows():
            x=read_gray(row.path); sid=str(row.source_id)
            if sid in ctx_cache: orders,br,plans=ctx_cache[sid]
            else: orders,br,plans=prepare_image_context(x,sid,risk,alpha,bs,seed)
            rr=run_frozen_image_precomputed(x,sid,primary_bpp,strategy,orders,br,bs,seed,plans=plans)
            if not rr['feasible']: continue
            y.extend([0,1]); scores.extend([det.score(x),det.score(rr['stego'])])
        score_arr=np.asarray(scores)
        m=detector_metrics(np.asarray(y),score_arr,float(config['detectors']['fixed_fpr']))
        m.update(paired_detector_bootstrap(score_arr[0::2],score_arr[1::2],float(config['detectors']['fixed_fpr']),n_resamples=int(config['statistics']['cluster_bootstrap_resamples']),confidence=float(config['statistics']['confidence']),seed=seed))
        m.update({'detector':'adaptive_srm_lite','strategy':strategy,'target_net_bpp':primary_bpp,'pairs':len(y)//2})
        adaptive_rows.append(m)
        print('adaptive',strategy,m)
    adaptive=pd.DataFrame(adaptive_rows)
    adaptive.to_csv(model_dir/'adaptive_srm_lite_primary_payload.csv',index=False)
    display(adaptive)

For a final submission, `SRM-lite` must be described by its actual feature construction; it must **not** be renamed “SRM”. If reviewer-grade comparison against full SRM/SRNet is needed, add those external implementations as an additional detector family without changing the frozen encoder/test split.